# Melanoma classifier — high-precision / high-F1 Swin V2 training

This notebook is based on the supplied Swin V2 training workflow and retains the checkpoint format used by this project. It is configured to **prioritize validation F1 or precision in the 95–98% range** only when that target is genuinely achieved. It never reports a target as achieved when the held-out validation data does not support it.

- validates image/label mapping and flags duplicate or patient leakage before training;
- uses lesion-preserving augmentation, mixed precision, gradient accumulation, warm-up, gradient clipping and EMA weights;
- applies **one** controlled class-imbalance strategy rather than stacking aggressive sampler and loss weights;
- selects the checkpoint using validation PR-AUC, and selects the clinical decision threshold on validation only;
- reports accuracy, balanced accuracy, precision, recall/sensitivity, specificity, NPV, F1/F2, MCC, Cohen's kappa, ROC-AUC, PR-AUC, Brier score and confidence intervals;
- evaluates the held-out test split once, after all choices are frozen.

For an imbalanced melanoma dataset, a very high precision can be produced by predicting very few positives. The precision option therefore also enforces a sensitivity floor, so the model remains useful for melanoma detection.


## Run order

1. Set `DATA_ROOT` in the configuration cell if Kaggle does not auto-detect the dataset.
2. Run through the data audit. Do not continue if cross-split duplicate images or patient leakage is reported.
3. Run training, then choose an operating point from the validation table.
4. Run the final test cell once. Do not use test results to tune hyperparameters or the threshold.


In [ ]:
!pip install -q timm==1.0.19 albumentations==2.0.8 scikit-learn seaborn

import contextlib, copy, hashlib, json, math, os, random, time, warnings
from collections import defaultdict
from pathlib import Path

import albumentations as A
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import timm
import torch
import torch.nn as nn
from albumentations.pytorch import ToTensorV2
from PIL import Image
from sklearn.calibration import calibration_curve
from sklearn.metrics import (accuracy_score, average_precision_score, balanced_accuracy_score,
    brier_score_loss, classification_report, cohen_kappa_score, confusion_matrix,
    f1_score, matthews_corrcoef, precision_score, recall_score, roc_auc_score,
    roc_curve, precision_recall_curve)
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=UserWarning)


In [ ]:
# Reproducibility and hardware -------------------------------------------------
SEED = 42
def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
seed_everything()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
torch.backends.cudnn.benchmark = True
if USE_AMP:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
print("Device:", DEVICE)
if USE_AMP:
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name}; VRAM: {p.total_memory / 2**30:.1f} GiB")


## Configuration

The model identifier and output checkpoint format match `src/classification/model.py` and `src/classification/inference.py` in this project.


In [ ]:
# Paths: the first existing candidate is used. Change DATA_ROOT explicitly if needed.
DATA_CANDIDATES = [
    Path("/kaggle/input/datasets/monish102006/melanoma-abcde-dataset-v1"),
    Path("/kaggle/input/melanoma-abcde-dataset-v1"),
    Path("dataset"),
]
DATA_ROOT = next((p for p in DATA_CANDIDATES if p.exists()), DATA_CANDIDATES[0])
OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs/training")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SPLIT_ALIASES = {"train": ("train",), "validation": ("validation", "val"), "test": ("test",)}
MODEL_NAME = "swinv2_base_window12to16_192to256"  # Must match project inference.
IMG_SIZE = 256
BATCH_SIZE = 8                 # increase only if GPU memory permits
GRAD_ACCUM_STEPS = 2           # effective batch = 16
NUM_WORKERS = min(4, max(0, (os.cpu_count() or 2) // 2))
MAX_EPOCHS = 35
FREEZE_BACKBONE_EPOCHS = 1
LR_HEAD, LR_BACKBONE = 1.5e-4, 1.5e-5
WEIGHT_DECAY = 1e-2
LABEL_SMOOTHING = 0.03
FOCAL_GAMMA = 1.25
EMA_DECAY = 0.999
PATIENCE = 8
MAX_GRAD_NORM = 1.0

# Use ONE imbalance treatment. sqrt_sampler is deliberately gentler than 50:50.
# Set to "none" when the source dataset is already balanced.
IMBALANCE_STRATEGY = "sqrt_sampler"  # "sqrt_sampler" | "none"
MONITOR = "pr_auc"                   # threshold-independent checkpoint selection
BEST_CHECKPOINT = OUTPUT_DIR / "best_swin_checkpoint.pth"
LATEST_CHECKPOINT = OUTPUT_DIR / "latest_swin_checkpoint.pth"
TRAINING_LOG = OUTPUT_DIR / "swin_training_log.csv"
RUN_FULL_HASH_AUDIT = True

assert DATA_ROOT.exists(), f"Dataset root not found: {DATA_ROOT}"
print("DATA_ROOT:", DATA_ROOT.resolve())
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())


## Resolve the supplied split metadata and audit the dataset

This uses the dataset's supplied train/validation/test split. It never reshuffles test images into training.


In [ ]:
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
IMAGE_COLUMNS = ("resolved_image_path", "image_path", "image_id", "image", "image_name", "filename", "file_name", "path", "file_path")
LABEL_COLUMNS = ("label", "class", "diagnosis", "target", "category", "class_name", "binary_label", "benign_malignant")
PATIENT_COLUMNS = ("patient_id", "patient", "patientid", "subject_id", "case_id")

def norm(x): return str(x).strip().lower().replace("-", "_").replace(" ", "_")
def first_column(frame, choices):
    by_norm = {norm(c): c for c in frame.columns}
    return next((by_norm[norm(c)] for c in choices if norm(c) in by_norm), None)
def find_split(name):
    return next((DATA_ROOT / alias for alias in SPLIT_ALIASES[name] if (DATA_ROOT / alias).is_dir()), None)
def metadata_path(name):
    candidates = [DATA_ROOT / "metadata" / f"{alias}.csv" for alias in SPLIT_ALIASES[name]]
    candidates += [DATA_ROOT / f"{alias}.csv" for alias in SPLIT_ALIASES[name]]
    return next((p for p in candidates if p.is_file()), None)

split_dirs = {name: find_split(name) for name in SPLIT_ALIASES}
csv_paths = {name: metadata_path(name) for name in SPLIT_ALIASES}
assert all(split_dirs.values()), f"Missing split directory: {split_dirs}"
assert all(csv_paths.values()), f"Missing metadata CSV: {csv_paths}"
metadata = {name: pd.read_csv(csv_paths[name]) for name in split_dirs}
print({name: {"dir": str(split_dirs[name]), "csv": str(csv_paths[name]), "rows": len(metadata[name])} for name in split_dirs})
print("Train columns:", metadata["train"].columns.tolist())

# Index files once; avoids a slow recursive search for every metadata row.
file_indexes = {}
for split, root in split_dirs.items():
    idx = defaultdict(list)
    image_count = 0
    for p in root.rglob("*"):
        if p.suffix.lower() in IMAGE_EXTS:
            image_count += 1
            idx[p.name].append(p)
            if p.stem != p.name: idx[p.stem].append(p)
    file_indexes[split] = idx
    print(f"{split}: indexed {image_count} image files")

def resolve_image(value, split, label):
    text = str(value).strip()
    candidates = [split_dirs[split] / text, split_dirs[split] / label / text]
    for candidate in candidates:
        if candidate.is_file(): return candidate
    hits = file_indexes[split].get(Path(text).name, [])
    if not hits: hits = file_indexes[split].get(Path(text).stem, [])
    if len(hits) == 1: return hits[0]
    if len(hits) > 1:
        labelled = [p for p in hits if norm(p.parent.name) == norm(label)]
        if len(labelled) == 1: return labelled[0]
    return None

records, patient_col = {}, None
for split, frame in metadata.items():
    image_col, label_col = first_column(frame, IMAGE_COLUMNS), first_column(frame, LABEL_COLUMNS)
    assert image_col and label_col, f"{split}: cannot identify image/label columns from {frame.columns.tolist()}"
    candidate_patient = first_column(frame, PATIENT_COLUMNS)
    patient_col = patient_col or candidate_patient
    rows = []
    for row_no, (_, row) in enumerate(frame.iterrows()):
        label = norm(row[label_col])
        path = resolve_image(row[image_col], split, label)
        if path is None: raise FileNotFoundError(f"{split} row {row_no}: cannot resolve {row[image_col]!r}")
        parent = norm(path.parent.name)
        if parent != label:
            raise ValueError(f"{split} row {row_no}: CSV label={label!r}, folder label={parent!r}, path={path}")
        rows.append({"path": path, "label_name": label, "patient_id": str(row[candidate_patient]) if candidate_patient else None})
    records[split] = pd.DataFrame(rows)

class_names = sorted(set().union(*(set(f.label_name) for f in records.values())))
assert len(class_names) == 2 and "melanoma" in class_names, f"Expected binary labels including melanoma; got {class_names}"
class_to_idx = {name: i for i, name in enumerate(class_names)}
melanoma_idx = class_to_idx["melanoma"]
print("class_to_idx:", class_to_idx)
for split, frame in records.items(): print(split, frame.label_name.value_counts().reindex(class_names, fill_value=0).to_dict())


In [ ]:
# Safety audit: duplicated patients and content-identical images across splits are leakage risks.
split_names = list(records)
for left_i, left in enumerate(split_names):
    for right in split_names[left_i + 1:]:
        lp = set(records[left].patient_id.dropna()) - {"None", "nan"}
        rp = set(records[right].patient_id.dropna()) - {"None", "nan"}
        overlap = lp & rp
        print(f"Patient overlap {left}/{right}: {len(overlap)}" + (f" e.g. {sorted(overlap)[:5]}" if overlap else ""))

def sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for part in iter(lambda: f.read(chunk), b""): h.update(part)
    return h.hexdigest()

if RUN_FULL_HASH_AUDIT:
    hashes = defaultdict(list)
    all_rows = pd.concat([f.assign(split=s) for s, f in records.items()], ignore_index=True)
    for row in tqdm(all_rows.itertuples(index=False), total=len(all_rows), desc="Hash audit"):
        hashes[sha256(row.path)].append((row.split, row.path.name, row.label_name))
    cross_split_duplicates = {h: rows for h, rows in hashes.items() if len({x[0] for x in rows}) > 1}
    print("Cross-split duplicate image groups:", len(cross_split_duplicates))
    if cross_split_duplicates:
        for _, rows in list(cross_split_duplicates.items())[:10]: print(rows)
        raise RuntimeError("Stop: remove/reassign cross-split duplicate images before training.")
print("Data audit passed. Continue only if the patient-overlap counts above are zero (or documented by the dataset).")


In [ ]:
# Distribution view
dist = pd.concat([f.assign(split=s) for s, f in records.items()])
ax = sns.countplot(data=dist, x="split", hue="label_name", order=["train", "validation", "test"])
ax.set_title("Supplied split class distribution")
plt.show()


## Datasets and lesion-preserving augmentation

Geometric/color changes are deliberately mild. There is no random augmentation in validation or test.


In [ ]:
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
PREPROCESSING_CONFIG = {"image_size": IMG_SIZE, "normalization": "ImageNet", "mean": MEAN, "std": STD,
                        "aspect_ratio": "preserved", "validation_transform": "letterbox -> ImageNet normalize"}
def letterbox(size):
    return [A.LongestMaxSize(max_size=size), A.PadIfNeeded(min_height=size, min_width=size, border_mode=cv2.BORDER_CONSTANT, fill=0)]
def transform(split):
    if split == "train":
        return A.Compose([*letterbox(IMG_SIZE), A.RandomResizedCrop(size=(IMG_SIZE, IMG_SIZE), scale=(0.82, 1.0), ratio=(0.9, 1.1)),
            A.HorizontalFlip(p=.5), A.VerticalFlip(p=.5), A.RandomRotate90(p=.5),
            A.Affine(scale=(.96, 1.04), translate_percent=(0, .04), rotate=(-10, 10), p=.3),
            A.OneOf([A.RandomBrightnessContrast(.12, .12, p=1), A.CLAHE(clip_limit=2.0, p=1)], p=.35),
            A.HueSaturationValue(8, 12, 8, p=.2), A.GaussNoise(std_range=(.02, .05), p=.08),
            A.Normalize(mean=MEAN, std=STD), ToTensorV2()])
    return A.Compose([*letterbox(IMG_SIZE), A.Normalize(mean=MEAN, std=STD), ToTensorV2()])

class MelanomaDataset(Dataset):
    def __init__(self, frame, tfm): self.frame, self.tfm = frame.reset_index(drop=True), tfm
    def __len__(self): return len(self.frame)
    def __getitem__(self, i):
        row = self.frame.iloc[i]
        image = np.asarray(Image.open(row.path).convert("RGB"))
        return self.tfm(image=image)["image"], torch.tensor(class_to_idx[row.label_name], dtype=torch.long)

train_ds, val_ds, test_ds = (MelanomaDataset(records[s], transform("train" if s == "train" else "eval")) for s in ("train", "validation", "test"))
loader_kw = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=USE_AMP, persistent_workers=NUM_WORKERS > 0)
if IMBALANCE_STRATEGY == "sqrt_sampler":
    counts = records["train"].label_name.value_counts()
    weights = records["train"].label_name.map(lambda x: 1 / math.sqrt(counts[x])).to_numpy(float)
    train_loader = DataLoader(train_ds, sampler=WeightedRandomSampler(weights, len(weights), replacement=True), drop_last=True, **loader_kw)
    print("Imbalance: sqrt-inverse sampler only; loss has no class alpha.")
elif IMBALANCE_STRATEGY == "none":
    train_loader = DataLoader(train_ds, shuffle=True, drop_last=True, **loader_kw)
else: raise ValueError("IMBALANCE_STRATEGY must be 'sqrt_sampler' or 'none'")
val_loader, test_loader = DataLoader(val_ds, shuffle=False, **loader_kw), DataLoader(test_ds, shuffle=False, **loader_kw)
print(f"Batches train/val/test: {len(train_loader)}/{len(val_loader)}/{len(test_loader)}")


## Model, optimizer, EMA, and loss

The wrapper is intentionally compatible with this project's Swin V2 inference module.


In [ ]:
class SwinV2Classifier(nn.Module):
    def __init__(self, model_name=MODEL_NAME, num_classes=2, pretrained=True, dropout_rate=.3):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=pretrained, num_classes=0)
        self.embed_dim = self.backbone.num_features
        self.head = nn.Sequential(nn.Dropout(dropout_rate), nn.Linear(self.embed_dim, num_classes))
    def forward(self, x):
        features = self.backbone(x)
        if features.ndim > 2: features = features.mean(dim=tuple(range(2, features.ndim)))
        return self.head(features)
    def get_backbone_params(self): return self.backbone.parameters()
    def get_head_params(self): return self.head.parameters()

class FocalLoss(nn.Module):
    def __init__(self, gamma=FOCAL_GAMMA, smoothing=LABEL_SMOOTHING): super().__init__(); self.gamma, self.smoothing = gamma, smoothing
    def forward(self, logits, target):
        ce = nn.functional.cross_entropy(logits, target, reduction="none", label_smoothing=self.smoothing)
        return (((1 - torch.exp(-ce)) ** self.gamma) * ce).mean()

class ModelEMA:
    def __init__(self, model, decay=EMA_DECAY): self.decay = decay; self.shadow = {k: v.detach().clone() for k,v in model.state_dict().items()}
    @torch.no_grad()
    def update(self, model):
        for k,v in model.state_dict().items():
            self.shadow[k] = v.detach().clone() if not torch.is_floating_point(v) else self.shadow[k].mul(self.decay).add(v.detach(), alpha=1-self.decay)
    def copy_to(self, model): model.load_state_dict(self.shadow, strict=True)

model = SwinV2Classifier().to(DEVICE)
criterion = FocalLoss()
optimizer = torch.optim.AdamW([{"params": model.get_backbone_params(), "lr": LR_BACKBONE}, {"params": model.get_head_params(), "lr": LR_HEAD}], weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS, eta_min=1e-7)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
ema = ModelEMA(model)
def amp(): return torch.autocast("cuda", dtype=torch.float16) if USE_AMP else contextlib.nullcontext()
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")


In [ ]:
# Metrics: melanoma is always the positive class, regardless of alphabetical class index.
def metrics(y_raw, p_melanoma, threshold=.5):
    y = (np.asarray(y_raw) == melanoma_idx).astype(int); p = np.asarray(p_melanoma); pred = (p >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    specificity = tn / (tn + fp) if tn + fp else np.nan
    npv = tn / (tn + fn) if tn + fn else np.nan
    precision = precision_score(y, pred, zero_division=0); recall = recall_score(y, pred, zero_division=0)
    return {"threshold": float(threshold), "accuracy": accuracy_score(y, pred), "balanced_accuracy": balanced_accuracy_score(y, pred),
            "precision": precision, "sensitivity": recall, "specificity": specificity, "npv": npv,
            "f1": f1_score(y, pred, zero_division=0), "f2": (5*precision*recall/(4*precision+recall)) if 4*precision+recall else 0.,
            "mcc": matthews_corrcoef(y, pred), "cohen_kappa": cohen_kappa_score(y, pred),
            "roc_auc": roc_auc_score(y, p), "pr_auc": average_precision_score(y, p), "brier": brier_score_loss(y, p),
            "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp), "confusion_matrix": np.array([[tn,fp],[fn,tp]])}

@torch.no_grad()
def predict(model_, loader, description="Evaluate"):
    model_.eval(); labels, probs, logits_all = [], [], []
    for images, target in tqdm(loader, desc=description, leave=False):
        with amp(): logits = model_(images.to(DEVICE, non_blocking=True))
        logits = logits.float().cpu(); logits_all.append(logits); labels.append(target.cpu()); probs.append(torch.softmax(logits, 1)[:, melanoma_idx])
    return np.concatenate(labels), np.concatenate(probs), np.concatenate(logits_all)

def threshold_table(y, p):
    rows = [metrics(y, p, t) for t in np.round(np.arange(.05, .951, .005), 3)]
    table = pd.DataFrame(rows)
    candidates = {"max_f1": table.loc[table.f1.idxmax()], "max_f2": table.loc[table.f2.idxmax()], "max_youden": table.loc[(table.sensitivity + table.specificity - 1).idxmax()]}
    for floor in (.80, .85, .90, .95):
        eligible = table[table.sensitivity >= floor]
        if len(eligible): candidates[f"sensitivity>={floor:.2f}"] = eligible.loc[eligible.specificity.idxmax()]
    return table, pd.DataFrame(candidates).T


In [ ]:
# Sanity check: run before the full training loop.
images, labels = next(iter(train_loader)); images, labels = images.to(DEVICE), labels.to(DEVICE)
with torch.no_grad(), amp(): logits = model(images); loss = criterion(logits, labels)
assert logits.shape == (len(labels), 2) and torch.isfinite(loss), (logits.shape, loss)
print("Sanity check passed:", tuple(images.shape), "loss=", float(loss))


## Train and resume

Set `RESUME_FROM = LATEST_CHECKPOINT` after an interruption. The best checkpoint uses EMA weights and is selected by validation PR-AUC, not accuracy.


In [ ]:
RESUME_FROM = None  # e.g. LATEST_CHECKPOINT
start_epoch, best_score, stale, history = 0, -np.inf, 0, []
if RESUME_FROM and Path(RESUME_FROM).is_file():
    ckpt = torch.load(RESUME_FROM, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"]); optimizer.load_state_dict(ckpt["optimizer_state_dict"]); scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    ema.shadow = {k: v.to(DEVICE) for k,v in ckpt["ema_state_dict"].items()}; start_epoch, best_score, stale, history = ckpt["epoch"], ckpt["best_score"], ckpt["stale"], ckpt.get("history", [])
    print("Resuming at epoch", start_epoch + 1)

def payload(epoch, score, stale_count):
    return {"epoch": epoch, "model_state_dict": {k:v.detach().cpu() for k,v in ema.shadow.items()}, "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(), "ema_state_dict": {k:v.detach().cpu() for k,v in ema.shadow.items()},
            "best_score": score, "stale": stale_count, "history": history, "class_names": class_names, "class_to_idx": class_to_idx,
            "model_config": {"model_name": MODEL_NAME, "num_classes": 2, "dropout": .3}, "preprocessing_config": PREPROCESSING_CONFIG,
            "imbalance_strategy": IMBALANCE_STRATEGY, "seed": SEED}

def set_backbone(trainable):
    for p in model.get_backbone_params(): p.requires_grad = trainable
set_backbone(start_epoch >= FREEZE_BACKBONE_EPOCHS)

for epoch in range(start_epoch, MAX_EPOCHS):
    if epoch == FREEZE_BACKBONE_EPOCHS: set_backbone(True); print("Backbone unfrozen.")
    started, train_loss, seen = time.time(), 0., 0
    model.train(); optimizer.zero_grad(set_to_none=True)
    for step, (images, labels) in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1:02d} train")):
        images, labels = images.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
        with amp(): loss = criterion(model(images), labels) / GRAD_ACCUM_STEPS
        scaler.scale(loss).backward()
        if (step + 1) % GRAD_ACCUM_STEPS == 0 or step + 1 == len(train_loader):
            scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True); ema.update(model)
        train_loss += loss.item() * GRAD_ACCUM_STEPS * len(labels); seen += len(labels)
    scheduler.step()
    raw_state = copy.deepcopy(model.state_dict()); ema.copy_to(model)
    vy, vp, _ = predict(model, val_loader, "Validation")
    val = metrics(vy, vp); model.load_state_dict(raw_state)
    score, improved = val[MONITOR], val[MONITOR] > best_score + 1e-4
    best_score, stale = (score, 0) if improved else (best_score, stale + 1)
    row = {"epoch": epoch+1, "train_loss": train_loss/max(seen,1), **{f"val_{k}": v for k,v in val.items() if k not in {"confusion_matrix"}}, "seconds": time.time()-started}
    history.append(row); pd.DataFrame(history).to_csv(TRAINING_LOG, index=False)
    torch.save(payload(epoch+1, best_score, stale), LATEST_CHECKPOINT)
    if improved: torch.save(payload(epoch+1, best_score, stale), BEST_CHECKPOINT)
    print(f"Epoch {epoch+1:02d}: loss={row['train_loss']:.4f}; val PR-AUC={val['pr_auc']:.4f}; ROC-AUC={val['roc_auc']:.4f}; F1@0.5={val['f1']:.4f}; best={best_score:.4f}")
    if stale >= PATIENCE: print("Early stopping."); break
print("Best checkpoint:", BEST_CHECKPOINT)


In [ ]:
# Training curves (a rapidly widening train/validation gap indicates overfitting).
history_df = pd.read_csv(TRAINING_LOG)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history_df.epoch, history_df.train_loss, label="train loss"); axes[0].set_title("Training loss"); axes[0].legend()
axes[1].plot(history_df.epoch, history_df.val_pr_auc, label="validation PR-AUC"); axes[1].plot(history_df.epoch, history_df.val_roc_auc, label="validation ROC-AUC"); axes[1].set_title("Validation ranking metrics"); axes[1].legend()
plt.show()


## Validation-only calibration check and threshold selection

Calibration does not change ROC-AUC/PR-AUC, but it reveals whether probabilities can be interpreted as risks. The deployment threshold below deliberately uses raw probabilities so it remains compatible with the existing project inference code.


In [ ]:
best = torch.load(BEST_CHECKPOINT, map_location=DEVICE, weights_only=False)
model.load_state_dict(best["model_state_dict"]); model.eval()
val_y, val_p, val_logits = predict(model, val_loader, "Best model / validation")

def expected_calibration_error(y_raw, p, bins=10):
    y = (np.asarray(y_raw) == melanoma_idx).astype(int); edges = np.linspace(0,1,bins+1); ece = 0.
    for lo, hi in zip(edges[:-1], edges[1:]):
        mask = (p >= lo) & (p < hi if hi < 1 else p <= hi)
        if mask.any(): ece += mask.mean() * abs(y[mask].mean() - p[mask].mean())
    return ece

print(f"Validation Brier={brier_score_loss((val_y==melanoma_idx).astype(int), val_p):.4f}; ECE={expected_calibration_error(val_y, val_p):.4f}")
frac_pos, mean_pred = calibration_curve((val_y == melanoma_idx).astype(int), val_p, n_bins=10, strategy="quantile")
plt.plot([0,1],[0,1],"--",color="gray"); plt.plot(mean_pred, frac_pos, "o-"); plt.xlabel("Mean predicted melanoma probability"); plt.ylabel("Observed melanoma rate"); plt.title("Validation reliability diagram"); plt.show()

thresholds, candidates = threshold_table(val_y, val_p)
show_cols = ["threshold", "accuracy", "balanced_accuracy", "precision", "sensitivity", "specificity", "f1", "f2", "mcc"]

# Choose the deployment priority. The requested 95–98% is a validation target,
# not a promise. Precision is protected by a sensitivity floor to avoid the
# trivial solution of predicting only one obvious melanoma.
PRIORITY_METRIC = "f1"                 # "f1" | "precision"
TARGET_MIN, TARGET_MAX = 0.95, 0.98
MIN_SENSITIVITY_FOR_PRECISION = 0.70   # only used when PRIORITY_METRIC="precision"

if PRIORITY_METRIC == "f1":
    target_rows = thresholds[thresholds.f1.between(TARGET_MIN, TARGET_MAX)]
    fallback_rows = thresholds
    fallback_label = "max_f1"
elif PRIORITY_METRIC == "precision":
    usable = thresholds[thresholds.sensitivity >= MIN_SENSITIVITY_FOR_PRECISION]
    target_rows = usable[usable.precision.between(TARGET_MIN, TARGET_MAX)]
    fallback_rows = usable
    fallback_label = f"max_precision_recall>={MIN_SENSITIVITY_FOR_PRECISION:.2f}"
else:
    raise ValueError("PRIORITY_METRIC must be 'f1' or 'precision'")

if len(target_rows):
    # F1 favors the most balanced result; precision priority uses F1 and then
    # sensitivity as tie-breakers, avoiding a tiny high-precision prediction set.
    order = [PRIORITY_METRIC, "f1", "sensitivity", "threshold"]
    target_best = target_rows.sort_values(order, ascending=False).iloc[0]
    SELECTION_STRATEGY = f"{PRIORITY_METRIC}_target_{TARGET_MIN:.2f}_to_{TARGET_MAX:.2f}"
    candidates.loc[SELECTION_STRATEGY] = target_best
    print(f"Target met: validation {PRIORITY_METRIC}={target_best[PRIORITY_METRIC]:.4f}.")
else:
    # Preserve honest reporting when the target cannot be attained on validation.
    target_best = fallback_rows.sort_values([PRIORITY_METRIC, "f1", "sensitivity", "threshold"], ascending=False).iloc[0]
    SELECTION_STRATEGY = fallback_label
    candidates.loc[SELECTION_STRATEGY] = target_best
    print(f"WARNING: validation {PRIORITY_METRIC} in [{TARGET_MIN:.2f}, {TARGET_MAX:.2f}] was not achieved. "
          f"Using the best validation {PRIORITY_METRIC} without falsifying the target.")

display(candidates[show_cols].sort_index().style.format("{:.4f}"))
SELECTED_THRESHOLD = float(candidates.loc[SELECTION_STRATEGY, "threshold"])
print("Frozen validation-selected threshold:", SELECTED_THRESHOLD, "strategy:", SELECTION_STRATEGY)


## One final held-out test evaluation

Run this only once after freezing the selected threshold. Its result is an estimate, not a parameter-search target.


In [ ]:
test_y, test_p, _ = predict(model, test_loader, "Final held-out test")
final = metrics(test_y, test_p, SELECTED_THRESHOLD)
print("Final test metrics")
for key in ("accuracy", "balanced_accuracy", "precision", "sensitivity", "specificity", "npv", "f1", "f2", "mcc", "cohen_kappa", "roc_auc", "pr_auc", "brier"):
    print(f"{key:>18}: {final[key]:.4f}")
print("Confusion matrix (rows=true non_melanoma/melanoma; cols=predicted non_melanoma/melanoma):\n", final["confusion_matrix"])
print(classification_report((test_y == melanoma_idx).astype(int), (test_p >= SELECTED_THRESHOLD).astype(int), target_names=["non_melanoma", "melanoma"], zero_division=0))

# Bootstrap CIs quantify uncertainty; especially useful when melanoma is rare.
rng = np.random.default_rng(SEED); ybin = (test_y == melanoma_idx).astype(int); boot = []
for _ in range(500):
    ix = rng.integers(0, len(ybin), len(ybin))
    if ybin[ix].min() == ybin[ix].max(): continue
    m = metrics(test_y[ix], test_p[ix], SELECTED_THRESHOLD)
    boot.append([m[k] for k in ("roc_auc", "pr_auc", "f1", "precision", "sensitivity", "specificity")])
ci = pd.DataFrame(boot, columns=["roc_auc", "pr_auc", "f1", "precision", "sensitivity", "specificity"]).quantile([.025, .975]).T
ci.columns = ["95% low", "95% high"]; display(ci.style.format("{:.4f}"))

fpr, tpr, _ = roc_curve(ybin, test_p); pr, rc, _ = precision_recall_curve(ybin, test_p)
fig, ax = plt.subplots(1, 2, figsize=(11,4)); ax[0].plot(fpr,tpr,label=f"AUROC={final['roc_auc']:.3f}"); ax[0].plot([0,1],[0,1],"--"); ax[0].set(xlabel="False positive rate",ylabel="True positive rate",title="Test ROC"); ax[0].legend()
ax[1].plot(rc,pr,label=f"AP={final['pr_auc']:.3f}"); ax[1].axhline(ybin.mean(),ls="--",color="gray",label=f"prevalence={ybin.mean():.3f}"); ax[1].scatter([final['sensitivity']],[final['precision']],color="red",label=f"threshold={SELECTED_THRESHOLD:.3f}"); ax[1].set(xlabel="Recall",ylabel="Precision",title="Test precision-recall"); ax[1].legend(); plt.show()


In [ ]:
# Save deployment metadata inside the same project-compatible checkpoint.
best["threshold"] = SELECTED_THRESHOLD
best["threshold_selection"] = {"strategy": SELECTION_STRATEGY, "selected_from": "validation_only", "candidates": candidates[show_cols].to_dict(orient="index")}
best["final_test_metrics"] = {k: (v.tolist() if isinstance(v, np.ndarray) else float(v) if isinstance(v, np.floating) else v) for k,v in final.items()}
best["training_log"] = history
torch.save(best, BEST_CHECKPOINT)
with open(OUTPUT_DIR / "final_test_metrics.json", "w") as f: json.dump(best["final_test_metrics"], f, indent=2)
print("Saved compatible checkpoint:", BEST_CHECKPOINT)
print("Saved metric report:", OUTPUT_DIR / "final_test_metrics.json")
print("Copy best_swin_checkpoint.pth to this project's checkpoints/ directory and set config.yaml paths.classification_checkpoint accordingly.")
